# Similarity Mejoras

## Libraries

In [1]:
import sys

print('Python version: ', sys.version)

Python version:  3.11.4 | packaged by conda-forge | (main, Jun 10 2023, 18:08:17) [GCC 12.2.0]


In [2]:
# Desactiva por completo la dependencia con TorchVision en Transformers
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"   # <- clave
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"      # opcional, menos ruido
os.environ["TRANSFORMERS_NO_TF"] = "1"       # <— evita que Transformers intente cargar TensorFlow
os.environ["TRANSFORMERS_NO_FLAX"] = "1"     # opcional, por si acaso

In [3]:
import site, sys
sys.path.insert(0, site.getusersitepackages())
print("usersite:", site.getusersitepackages())

usersite: /home/jovyan/.local/lib/python3.11/site-packages


In [4]:
!pip uninstall -y sentence-transformers transformers tokenizers torchvision

Found existing installation: sentence-transformers 2.6.1
Uninstalling sentence-transformers-2.6.1:
  Successfully uninstalled sentence-transformers-2.6.1
Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3


In [5]:
# Instala versiones probadas para Python 3.11 (sin tocar torch/cuda del sistema)
# NOTA: usamos --user para no requerir root; --no-cache-dir por si hay ruedas viejas cacheadas
!pip install --user --upgrade --no-cache-dir \
  "tokenizers==0.20.3" \
  "transformers==4.46.3" \
  "sentence-transformers==2.6.1" \
  "scikit-learn==1.5.2" \
  "pandas==2.2.3" \
  "pyarrow==17.0.0" \
  "numexpr>=2.10.1" \
  "bottleneck>=1.4.0"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 52.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 110.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 422.6 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import numpy, pandas, sklearn, transformers, tokenizers, sentence_transformers, torch, pyarrow, numexpr, bottleneck
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| cuda ver:", torch.version.cuda)
print("pyarrow:", pyarrow.__version__)
print("numexpr:", numexpr.__version__)
print("bottleneck:", bottleneck.__version__)

numpy: 2.3.2
pandas: 2.2.3
sklearn: 1.5.2
transformers: 4.46.3 | tokenizers: 0.20.3
sentence-transformers: 2.6.1
torch: 2.8.0+cu128 | cuda: True | cuda ver: 12.8
pyarrow: 17.0.0
numexpr: 2.11.0
bottleneck: 1.5.0


In [7]:
import types

if "torchvision" not in sys.modules:
    tv = types.ModuleType("torchvision")
    tv_t = types.ModuleType("torchvision.transforms")
    class _InterpolationMode: pass
    tv_t.InterpolationMode = _InterpolationMode
    tv.transforms = tv_t
    sys.modules["torchvision"] = tv
    sys.modules["torchvision.transforms"] = tv_t

# (Opcional) Si por alguna razón Transformers todavía cree que hay TF, fuerza a falso:
try:
    import transformers.utils.import_utils as _iu
    _iu.is_tf_available = lambda: False
except Exception:
    pass

print("Guardia OK. usersite first:", sys.path[0])

Guardia OK. usersite first: /home/jovyan/.local/lib/python3.11/site-packages


In [8]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

emb = model.encode(["hola mundo"], convert_to_tensor=True)
print(emb.shape, emb.device)


torch.Size([1, 384]) cuda:0


## A) Setup básico y rutas portables 

In [9]:
import os, re, math, string, random
import numpy as np
import pandas as pd
import torch

# Semillas reproducibles
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# Base portátil: directorio del notebook (si existiese env var BASE, úsala)
BASE = os.environ.get("BASE", os.getcwd())
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [10]:
from pprint import pprint

# Carga DWAs
df_onet = pd.read_csv(os.path.join(DATA_DIR, "dwas.csv"))
onetLabels = df_onet["dwa_title"].astype(str).tolist()

pprint(onetLabels)
print(len(onetLabels))

['Train personnel on proper operational procedures.',
 'Prepare procedural documents.',
 'Recommend technical design or process changes to improve efficiency, '
 'quality, or performance.',
 'Confer with technical personnel to prepare designs or operational plans.',
 'Communicate technical information to suppliers, contractors, or regulatory '
 'agencies.',
 'Design medical devices or appliances.',
 'Research engineering aspects of biological or chemical processes.',
 'Devise research or testing protocols.',
 'Develop operational methods or processes that use green materials or '
 'emphasize sustainability.',
 'Develop technical methods or processes.',
 'Create models of engineering designs or methods.',
 'Maintain operational records or records systems.',
 'Supervise engineering or other technical personnel.',
 'Estimate operational costs.',
 'Estimate time requirements for development or production projects.',
 'Prepare detailed work plans.',
 'Prepare technical reports for internal 

## B) Utils: limpieza y funciones auxiliares

In [11]:
BASIC_STOP = {
    "and","or","to","of","the","a","an","with","for","in","on","by","from",
    "at","as","into","than","that","this","those","these","is","are","be"
}

_punct_tbl = str.maketrans({c:" " for c in string.punctuation})

def normalize_text(s: str) -> str:
    s = s.lower().translate(_punct_tbl)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s: str):
    return [t for t in normalize_text(s).split() if t]

def jaccard_stopwords(a: str, b: str, stop=BASIC_STOP):
    A = {t for t in tokenize(a) if t in stop}
    B = {t for t in tokenize(b) if t in stop}
    if not A and not B: return 0.0
    return len(A & B) / len(A | B)

def length_penalty(a: str, b: str):
    la, lb = len(tokenize(a)), len(tokenize(b))
    if max(la, lb) == 0: return 0.0
    return abs(la - lb) / max(la, lb)


## C) Model loader & encoding

In [12]:
from sentence_transformers import SentenceTransformer

_MODEL_CACHE = {}

def get_model(name_model: str) -> SentenceTransformer:
    if name_model not in _MODEL_CACHE:
        _MODEL_CACHE[name_model] = SentenceTransformer(name_model)
    return _MODEL_CACHE[name_model]

@torch.no_grad()
def encode_texts(texts, name_model: str, device=None) -> torch.Tensor:
    m = get_model(name_model)
    embs = m.encode(texts, convert_to_tensor=True, device=device)
    # normalizamos a norma 1 para usar producto como coseno
    embs = torch.nn.functional.normalize(embs, p=2, dim=1)
    return embs


## D) Búsqueda de vecinos con normalización local + filtros

In [13]:
@torch.no_grad()
def nearest_neighbors_dwa(
    labels,                     # lista de DWAs (strings)
    name_model: str,
    k: int = 10,                # top-k inicial (antes de filtros)
    k_return: int = 3,          # cuántos devolver tras re-ranking
    tau_z: float = 0.0,         # umbral sobre z-score local (>=0 ≈ por encima de la media local)
    lam_len: float = 0.10,      # peso penalización longitud
    rho_stop: float = 0.20,     # peso penalización solapamiento stopwords
    device=None
):
    n = len(labels)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    E = encode_texts(labels, name_model, device=device)  # [n,d], unit norm
    # Similaridad coseno como producto escalar
    S = (E @ E.T).float()  # [n,n]
    # Excluir self-match
    S.fill_diagonal_(-1.0)

    # top-k candidatos por fila
    topk_vals, topk_idx = torch.topk(S, k=min(k, n-1), dim=1)  # [n,k]

    # --- Normalización local (z-score por fila) ---
    mu = S.mean(dim=1, keepdim=True)          # [n,1]
    sd = S.std(dim=1, keepdim=True).clamp_min(1e-6)
    Z = (S - mu) / sd                         # [n,n]
    z_topk = torch.gather(Z, 1, topk_idx)     # [n,k]

    # --- MNN (mutual nearest neighbors) ---
    # Calculamos lista inversa: para cada j, sus top-k también
    # (ya la tenemos en topk_idx/z_topk)
    # Creamos un set de pares mutuos (i,j)
    mnn = set()
    inv_lists = [set(topk_idx[j].tolist()) for j in range(n)]
    for i in range(n):
        for j in topk_idx[i].tolist():
            if i in inv_lists[j]:
                mnn.add((i, j))

    # --- Re-ranking con penalizaciones ---
    results = []
    for i in range(n):
        cand_idx = topk_idx[i].tolist()
        base_sims = topk_vals[i].tolist()
        base_zs   = z_topk[i].tolist()
        rows = []
        for j, s_ij, z_ij in zip(cand_idx, base_sims, base_zs):
            if z_ij < tau_z:
                continue  # umbral sobre z-score local
            # penalizaciones
            lp = length_penalty(labels[i], labels[j])          # [0,1]
            js = jaccard_stopwords(labels[i], labels[j])       # [0,1] típicamente pequeño
            # score combinado (restamos penalizaciones)
            score = z_ij - lam_len*lp - rho_stop*js
            rows.append({
                "i": i, "j": j,
                "neighbor": labels[j],
                "sim": float(s_ij),
                "z_sim": float(z_ij),
                "len_pen": float(lp),
                "stop_jacc": float(js),
                "score": float(score),
                "mnn": (i, j) in mnn
            })
        # Orden final por score, priorizando MNN
        rows.sort(key=lambda r: (not r["mnn"], -r["score"]))
        results.append(rows[:k_return])

    # Construimos DataFrame estilo anterior (label2..4), ahora con métricas extra
    recs = []
    for i in range(n):
        row = {"originalSkill": labels[i]}
        for rnk, item in enumerate(results[i], start=1):
            row[f"label{rnk}"] = item["neighbor"]
            row[f"sim{rnk}"]   = item["sim"]
            row[f"z{rnk}"]     = item["z_sim"]
            row[f"mnn{rnk}"]   = item["mnn"]
            row[f"score{rnk}"] = item["score"]
        recs.append(row)
    df_out = pd.DataFrame(recs)
    return df_out


## E) Ejecutar para varios modelos y consolidar

In [14]:
model_names = [
    "sentence-transformers/paraphrase-albert-small-v2",
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/distiluse-base-multilingual-cased-v2",
    "sentence-transformers/sentence-t5-base",
    "sentence-transformers/all-distilroberta-v1",
    "embedding-data/deberta-sentence-transformer",
    "sentence-transformers/paraphrase-MiniLM-L3-v2",
    "sentence-transformers/all-mpnet-base-v2"
]

df_skills_general = pd.DataFrame()
for name_model in model_names:
    df_nn = nearest_neighbors_dwa(
        labels=onetLabels,
        name_model=name_model,
        k=20,          # top-k inicial para filtros
        k_return=3,    # 3 alternativas finales
        tau_z=0.0,     # empieza conservador; luego ajustaremos
        lam_len=0.10,
        rho_stop=0.20
    )
    df_nn["model"] = name_model
    df_nn["data"]  = "onet"
    # Guarda por modelo (CSV; más ligero que Excel)
    safe_model_name = name_model.replace("/", "_").replace(":", "_")
    df_nn.to_csv(os.path.join(RESULTS_DIR, f"neighbors_{safe_model_name}_onet.csv"), index=False)
    df_skills_general = pd.concat([df_skills_general, df_nn], ignore_index=True)

# Vista consolidada rápida
df_skills_general.head(5)


,originalSkill,label1,sim1,z1,mnn1,score1,label2,sim2,z2,mnn2,score2,label3,sim3,z3,mnn3,score3,model,data
0,Train personnel on proper operational procedures.,Recruit personnel.,0.556166,1.881492,True,1.814825,Hire personnel.,0.534524,1.725257,True,1.658591,Investigate industrial or transportation accid...,0.456770,1.163947,True,1.147280,sentence-transformers/paraphrase-albert-small-v2,onet
1,Prepare procedural documents.,Prepare proposal documents.,0.607411,2.128168,True,2.128168,Review technical documents to plan work.,0.606049,2.118657,True,2.068657,Advise customers on technical or procedural is...,0.595470,2.044808,True,1.987665,sentence-transformers/paraphrase-albert-small-v2,onet
2,Recommend technical design or process changes ...,Implement design or process improvements.,0.762982,2.733440,True,2.575107,Recommend organizational process or policy cha...,0.714835,2.400423,True,2.250423,Identify opportunities to improve operational ...,0.623981,1.772017,True,1.622017,sentence-transformers/paraphrase-albert-small-v2,onet
3,Confer with technical personnel to prepare des...,Confer with other personnel to resolve design ...,0.850888,2.786227,True,2.586227,Provide technical guidance to other personnel.,0.726670,2.020856,True,1.914190,Confer with organizational members to accompli...,0.705030,1.887527,True,1.734194,sentence-transformers/paraphrase-albert-small-v2,onet
4,Communicate technical information to suppliers...,"Coordinate activities with suppliers, contract...",0.657485,2.122287,True,2.055620,Communicate organizational information to cust...,0.671705,2.221565,True,2.010453,Confer with technical personnel to prepare des...,0.645751,2.040363,True,1.897030,sentence-transformers/paraphrase-albert-small-v2,onet


## F) Evaluación & Calibración 

In [15]:
import numpy as np
import pandas as pd
import torch
from typing import List, Dict, Tuple

# 1) Gold standard (puedes editarlo luego; empieza con algo manejable)
GOLD: Dict[str, Dict[str, List[str]]] = {
    "Train personnel on proper operational procedures.": {
        "positives": [
            "Conduct employee training programs.",
            "Instruct college students in physical or life sciences."
        ],
        "negatives": [
            "Prepare proposal documents.",
            "Purchase materials, equipment, or other resources."
        ]
    },
    "Prepare procedural documents.": {
        "positives": [
            "Document organizational or operational procedures.",
            "Prepare contracts, disclosures, or applications."
        ],
        "negatives": [
            "Operate industrial equipment.",
            "Recruit personnel."
        ]
    },
    "Design medical devices or appliances.": {
        "positives": [
            "Design electromechanical equipment or systems.",
            "Design industrial equipment."
        ],
        "negatives": [
            "Develop business or financial information systems.",
            "Supervise employees."
        ]
    },
    "Conduct environmental audits.": {
        "positives": [
            "Monitor activities affecting environmental quality.",
            "Analyze environmental regulations to ensure organizational compliance."
        ],
        "negatives": [
            "Direct sales, marketing, or customer service activities.",
            "Interview employees, customers, or others to collect information."
        ]
    },
    "Develop operational methods or processes that use green materials or emphasize sustainability.": {
        "positives": [
            "Develop sustainable business strategies or practices.",
            "Implement transportation changes to reduce environmental impact."
        ],
        "negatives": [
            "Manage human resources activities.",
            "Fabricate devices or components."
        ]
    },
    "Estimate operational costs.": {
        "positives": [
            "Estimate labor requirements.",
            "Estimate time requirements for development or production projects."
        ],
        "negatives": [
            "Research genetic characteristics or expression.",
            "Inspect finished products to locate flaws."
        ]
    },
    "Supervise engineering or other technical personnel.": {
        "positives": [
            "Supervise production or support personnel.",
            "Supervise scientific or technical personnel."
        ],
        "negatives": [
            "Prepare financial documents.",
            "Analyze chemical compounds or substances."
        ]
    },
    "Research engineering aspects of biological or chemical processes.": {
        "positives": [
            "Research microbiological or chemical processes or structures.",
            "Research engineering applications of emerging technologies."
        ],
        "negatives": [
            "Manage inventories of products or organizational resources.",
            "Negotiate labor disputes."
        ]
    },
    "Develop software or computer applications.": {
        "positives": [
            "Develop business or financial information systems.",
            "Program robotic equipment."
        ],
        "negatives": [
            "Hire personnel.",
            "Administer standardized physical or psychological tests."
        ]
    },
    "Evaluate quality of materials or products.": {
        "positives": [
            "Test quality of materials or finished products.",
            "Inspect finished products to locate flaws."
        ],
        "negatives": [
            "Direct organizational operations, projects, or services.",
            "Perform marketing activities."
        ]
    }
}


In [16]:
# 2) Configura aquí los modelos candidatos (nombres HF o claves de tu caché)
CANDIDATE_MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    # añade otros que estés probando…
]

TOPK_LIST = [1, 3, 5, 10]
THRESHOLDS = np.round(np.linspace(0.4, 0.9, 11), 2)  # barrido para calibrar

In [17]:
# 3) Utilidades reutilizando tus funciones existentes (ajusta nombres si difieren)
def embed_texts(model, texts: List[str], batch_size: int = 64, to_tensor=True):
    emb = model.encode(texts, batch_size=batch_size, convert_to_tensor=to_tensor, normalize_embeddings=True)
    return emb

def cosine_topk(query_emb: torch.Tensor, corpus_emb: torch.Tensor, k: int = 10
               ) -> Tuple[torch.Tensor, torch.Tensor]:
    # query_emb: (d,) o (1, d); corpus_emb: (N, d)
    if query_emb.dim() == 1:
        query_emb = query_emb.unsqueeze(0)
    sims = torch.matmul(query_emb, corpus_emb.T)  # cos si ya normalizaste
    topk_sims, topk_idx = torch.topk(sims, k=min(k, corpus_emb.size(0)), dim=1)
    return topk_idx[0], topk_sims[0]

def precision_recall_at_k(ranked_labels: List[int], k: int, num_positives: int) -> Tuple[float, float]:
    hits = sum(ranked_labels[:k])
    prec = hits / k
    rec = hits / max(1, num_positives)
    return prec, rec

def evaluate_one_model(model_name: str,
                       dwas_all: List[str],
                       gold: Dict[str, Dict[str, List[str]]],
                       topk_list = TOPK_LIST,
                       thresholds = THRESHOLDS,
                       device = "cuda" if torch.cuda.is_available() else "cpu"):
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(model_name, device=device)
    # Indexamos TODO el corpus una vez
    corpus_texts = dwas_all
    corpus_emb = embed_texts(model, corpus_texts)  # (N, d) tensor normalizado

    # mapa rápido texto->idx
    idx_of = {t: i for i, t in enumerate(corpus_texts)}

    rows = []
    thr_rows = []

    for query_text, sets in gold.items():
        if query_text not in idx_of:
            # si la query no está en el corpus, sáltala (o añádela al corpus antes)
            continue

        q_idx = idx_of[query_text]
        q_emb = corpus_emb[q_idx]

        # preparamos etiquetas de verdad
        positives = set([p for p in sets.get("positives", []) if p in idx_of])
        negatives = set([n for n in sets.get("negatives", []) if n in idx_of])
        pos_idx = {idx_of[p] for p in positives}
        neg_idx = {idx_of[n] for n in negatives}
        num_pos = len(pos_idx)

        # recupera top- K (usamos el mayor K para una sola vez)
        maxK = max(topk_list)
        top_idx, top_sims = cosine_topk(q_emb, corpus_emb, k=maxK+1)  # +1 porque quitarás la propia query
        # elimina la propia query si aparece
        filtered = [(i.item(), s.item()) for i, s in zip(top_idx, top_sims) if i.item() != q_idx]
        top_idx, top_sims = zip(*filtered[:maxK]) if filtered else ([], [])

        # vector de labels (1 si es positive, 0 si no)
        ranked_labels = [1 if i in pos_idx else 0 for i in top_idx]

        for k in topk_list:
            prec, rec = precision_recall_at_k(ranked_labels, min(k, len(ranked_labels)), num_pos)
            rows.append({
                "model": model_name,
                "query": query_text,
                "k": k,
                "precision@k": prec,
                "recall@k": rec,
                "num_pos": num_pos
            })

        # barrido de umbral (clasificador binario basado en similitud)
        sims_arr = np.array(top_sims)
        labels_arr = np.array(ranked_labels)
        for thr in thresholds:
            preds = (sims_arr >= thr).astype(int)
            tp = int(np.sum((preds == 1) & (labels_arr == 1)))
            fp = int(np.sum((preds == 1) & (labels_arr == 0)))
            fn = int(np.sum((preds == 0) & (labels_arr == 1)))
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
            thr_rows.append({
                "model": model_name,
                "query": query_text,
                "threshold": float(thr),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "tp": tp, "fp": fp, "fn": fn,
                "num_pos": num_pos
            })

    df_atk = pd.DataFrame(rows)
    df_thr = pd.DataFrame(thr_rows)

    # agregados por modelo
    atk_summary = (df_atk.groupby(["model","k"])
                     .agg(
                         precision_mean=("precision@k","mean"),
                         recall_mean=("recall@k","mean")
                     )
                     .reset_index())

    thr_summary = (df_thr.groupby(["model","threshold"])
                         .agg(precision=("precision","mean"),
                              recall=("recall","mean"),
                              f1=("f1","mean"))
                         .reset_index())

    # mejor umbral por F1 medio
    best_thr_idx = thr_summary.groupby("model")["f1"].idxmax()
    best_thr = thr_summary.loc[best_thr_idx].reset_index(drop=True)

    return df_atk, atk_summary, df_thr, thr_summary, best_thr


In [18]:
# 4) Ejecuta la evaluación sobre tus DWAs completas (ajusta variable con tu lista total)
DWAS_ALL = list(dwas_df["dwa"].unique()) if "dwas_df" in globals() and "dwa" in dwas_df.columns else onetLabels

all_results = []
atk_summaries = []
thr_summaries = []
best_thrs = []

for m in CANDIDATE_MODELS:
    df_atk, atk_sum, df_thr, thr_sum, best_thr = evaluate_one_model(m, DWAS_ALL, GOLD)
    all_results.append(df_atk)
    atk_summaries.append(atk_sum)
    thr_summaries.append(thr_sum)
    best_thrs.append(best_thr)

DF_ATK = pd.concat(all_results, ignore_index=True)
ATK_SUMMARY = pd.concat(atk_summaries, ignore_index=True)
THR_SUMMARY = pd.concat(thr_summaries, ignore_index=True)
BEST_THR = pd.concat(best_thrs, ignore_index=True).sort_values("f1", ascending=False).reset_index(drop=True)

print(">> Resumen precision/recall@k")
display(ATK_SUMMARY.sort_values(["k","precision_mean"], ascending=[True, False]))

print("\n>> Umbral óptimo por F1 (medio en queries del gold)")
display(BEST_THR)

>> Resumen precision/recall@k


,model,k,precision_mean,recall_mean
0,sentence-transformers/all-MiniLM-L6-v2,1,0.400000,0.20
4,sentence-transformers/paraphrase-multilingual-...,1,0.400000,0.20
1,sentence-transformers/all-MiniLM-L6-v2,3,0.333333,0.50
5,sentence-transformers/paraphrase-multilingual-...,3,0.300000,0.45
2,sentence-transformers/all-MiniLM-L6-v2,5,0.260000,0.65
6,sentence-transformers/paraphrase-multilingual-...,5,0.220000,0.55
3,sentence-transformers/all-MiniLM-L6-v2,10,0.140000,0.70
7,sentence-transformers/paraphrase-multilingual-...,10,0.120000,0.60



>> Umbral óptimo por F1 (medio en queries del gold)


,model,threshold,precision,recall,f1
0,sentence-transformers/all-MiniLM-L6-v2,0.55,0.343016,0.70,0.408182
1,sentence-transformers/paraphrase-multilingual-...,0.65,0.246190,0.65,0.321349


## G) Reporte & Análisis de errores

In [19]:
import pandas as pd

# 1) Elige el modelo ganador y su umbral recomendado
if len(BEST_THR) == 0:
    raise RuntimeError("No hay BEST_THR; ejecuta antes la celda F.")
WINNER_MODEL = BEST_THR.iloc[0]["model"]
WINNER_THR = float(BEST_THR.iloc[0]["threshold"])
WINNER_F1 = float(BEST_THR.iloc[0]["f1"])

print(f"Modelo recomendado: {WINNER_MODEL}")
print(f"Umbral recomendado: {WINNER_THR:.2f}  (F1 medio={WINNER_F1:.3f})")

# 2) Tabla compacta para informe
report_table = (ATK_SUMMARY[ATK_SUMMARY["model"]==WINNER_MODEL]
                .sort_values("k"))
display(report_table)

# 3) Errores más comunes del ganador (falsos positivos por encima del umbral)
#    (Requiere que hayas guardado DF_ATK y THR_SUMMARY del modelo ganador)
winner_thr = THR_SUMMARY[THR_SUMMARY["model"]==WINNER_MODEL].copy()
near_thr = winner_thr[winner_thr["threshold"].between(WINNER_THR-0.01, WINNER_THR+0.01)]
display(near_thr.sort_values("f1", ascending=False).head(10))

print("\nNOTA:")
print("- Usa este modelo y umbral en el bloque de producción (antes llamado F/G).")
print("- Si priorizas PRECISIÓN alta (pocos falsos positivos), sube el umbral ~+0.02–0.05.")
print("- Si priorizas RECALL (no perder positivos), bájalo ~-0.02–0.05 y limita k.")


Modelo recomendado: sentence-transformers/all-MiniLM-L6-v2
Umbral recomendado: 0.55  (F1 medio=0.408)


,model,k,precision_mean,recall_mean
0,sentence-transformers/all-MiniLM-L6-v2,1,0.400000,0.20
1,sentence-transformers/all-MiniLM-L6-v2,3,0.333333,0.50
2,sentence-transformers/all-MiniLM-L6-v2,5,0.260000,0.65
3,sentence-transformers/all-MiniLM-L6-v2,10,0.140000,0.70


,model,threshold,precision,recall,f1
3,sentence-transformers/all-MiniLM-L6-v2,0.55,0.343016,0.7,0.408182



NOTA:
- Usa este modelo y umbral en el bloque de producción (antes llamado F/G).
- Si priorizas PRECISIÓN alta (pocos falsos positivos), sube el umbral ~+0.02–0.05.
- Si priorizas RECALL (no perder positivos), bájalo ~-0.02–0.05 y limita k.
